# Kaggle × Colab validation

Template for checking experiments on Colab without using a Kaggle submission. It runs `competitions/<COMPETITION>/validate.py`, which each competition provides.

Set `KAGGLE_API_TOKEN` in 🔑 Secrets, edit the options (`DRIVE_DIR` is the competition's folder in My Drive), then run all. Start with `--num-tasks 2` to check the setup and speed, then remove it. Running again continues from the results on Drive.

In [ ]:
# @title Options
COMPETITION = "gemma-4-developer-agent"  # @param {type:"string"}
# comma-separated; the first is the baseline
EXPS = "exp001,exp003"  # @param {type:"string"}
ARGS = "--num-tasks 2"  # @param {type:"string"}
# folder in My Drive for this competition: results go here, adapters are read from <exp>/ here
DRIVE_DIR = "Colab Notebooks/Kaggle/2026_Gemma4"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}


In [ ]:
# @title Setup
import importlib
import os
import sys

from google.colab import drive, userdata

REPO = "Moriyuki-S/kaggle_ws"
REPO_DIR = "/content/kaggle_ws"

for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "GITHUB_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass

# results and adapters live on Drive so that they survive a disconnect
drive.mount("/content/drive")
folder = DRIVE_DIR.strip().strip("/")
for prefix in ("content/drive/MyDrive/", "My Drive/", "マイドライブ/"):
    folder = folder.removeprefix(prefix)
OUTPUT_DIR = f"/content/drive/MyDrive/{folder}"
if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(f"No such Drive folder: {OUTPUT_DIR}")
os.environ["OUTPUT_DIR"] = OUTPUT_DIR  # read by kaggle_ws.get_paths in validate.py
print("Drive folder:", OUTPUT_DIR, sorted(os.listdir(OUTPUT_DIR)))

token = os.environ.get("GITHUB_TOKEN")
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.exists(REPO_DIR):
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH} && git log -1 --oneline

!pip install -q -U kaggle && pip install -q --no-deps -e .


In [ ]:
# @title Run
!python competitions/{COMPETITION}/validate.py --exps {EXPS} {ARGS}
exit_code = _exit_code  # noqa: F821  (set by IPython after a ! command)
if exit_code != 0:
    raise RuntimeError(f"validate.py failed (exit code {exit_code})")
